# Environment

## Import Library

In [37]:
#import torch
#from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import pandas as pd
import json
import os
import time
from openai import AzureOpenAI
from dotenv import load_dotenv



# load_azure_openai_client

In [56]:
# Load environment variables from .env file
def load_azure_openai_client():
    load_dotenv()
    return AzureOpenAI(
        api_version=os.getenv('api_version'),
        azure_endpoint=os.getenv('api_endpoint'),
        api_key=os.getenv('api_key'),
    )


### GPT4.1


In [57]:
def run_inference(prompt, model_deployment_name="gpt-4.1"):
    client = load_azure_openai_client()

    response = client.chat.completions.create(
        model=model_deployment_name,  # this must match the deployment name in Azure
        messages=[
            {"role": "user", "content": prompt}
        ],
        seed=42,
        max_tokens=256,
   
    )

    return response.choices[0].message.content


In [58]:
result = run_inference("give me a cloud logs?")
print("Response:", result)

Response: Sure! Here is a generic example of what **cloud logs** might look like in a typical cloud environment. Cloud logs record events, errors, warnings, and informational messages about resources and applications running in the cloud.

---

**Example Cloud Log Entries:**

```
2024-06-10T08:35:22Z [INFO] Instance i-0a1b2c3d4e5f6g7h8 started in region us-east-1
2024-06-10T08:35:25Z [WARNING] Failed login attempt from IP 192.168.1.45 to instance i-0a1b2c3d4e5f6g7h8
2024-06-10T08:35:28Z [ERROR] Application 'web-app-v2' crashed with error code 502
2024-06-10T08:36:10Z [INFO] S3 bucket 'prod-data-bucket' accessed by user 'alice'
2024-06-10T08:36:22Z [INFO] Autoscaling triggered: launching new instance i-1a2b3c4d5e6f7g8h9
2024-06


# Load Datasets : dpo

In [77]:
import pandas as pd
cloud_dpo=pd.read_csv("dpo_cloud_train_test_combine_data.csv")
#device_dpo=pd.read_csv("data/dpo_device_train.csv")

In [80]:
cloud_dpo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2349 entries, 0 to 2348
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  2349 non-null   object
 1   chosen    2349 non-null   object
 2   rejected  2349 non-null   object
dtypes: object(3)
memory usage: 55.2+ KB


In [82]:
cloud_dpo.head()

,question,chosen,rejected
0,OCR Text:\nimg0:\nNon-table text:\n ...,A service connection is a connector to an exte...,No answer
1,OCR Text:\nimg0:\nNon-table text:\n ...,\r\nYour error cognito-idp:ListUsers is about ...,The issue here seems that your role does not h...
2,OCR Text:\nimg0:\nNon-table text:\n ...,"\r\nAccording to this doc about agent pools, t...",The problem is that your yaml file has no refe...
3,OCR Text:\nimg0:\nNon-table text:\n ...,My guess here is that this only happens infreq...,The issue here seems like your Lambda function...
4,OCR Text:\nimg0:\nNon-table text:\n ...,May be Your passed AWS configure region is dif...,The issue here seems like there's something wr...


In [83]:
print(cloud_dpo['rejected'].iloc[0])

No answer


In [84]:
#print(device_dpo['rejected'].iloc[43])

In [85]:
print(cloud_dpo['chosen'].loc[10])


There are two settings here that affect the producer:


acks - this is a producer-level setting
min.insync.replicas - this is a topic-level setting


The acks property determines how you want to handle writing to kafka:

acks=0 - I don't care about receiving acknowledgment of receipt


acks=1 - Send an acknowledgment when the leader partition has received the batch in memory


all/-1 - Wait for all replicas to receive the batch before sending an acknowledgment 


Keep in mind, the receipt in the partition is in memory, Kafka by default doesn't wait for fsync to disk, so acks=1 is not a durable write!

min.insync.replicas is used when there is a problem in the topic, maybe one of the partitions is not in-sync, or offline. When this is the case the cluster will send an ack when min.insync.replicas is satisfied. So 3 replicas, with min.insync.replicas=2 will still be able to write:


The acks property has no affect on the consumers, just that the data won't be written until acks and min.

In [86]:
cloud_dpo.loc[10]

question    OCR Text:\nimg0:\nNon-table text:\n           ...
chosen      \r\nThere are two settings here that affect th...
rejected    This property has no effect on consumer side.\...
Name: 10, dtype: object

# Prompt

## Cloud

### Chosen Prompt -> Human Answer

In [87]:
choosen_explanation_prompt = '''
Given a question and its selected answer, provide a brief explanation of why this answer is correct and appropriately addresses the question.

Question:
{}

Chosen Answer:
{}

Explanation:
'''

In [88]:
def choosen_explanation_prompt_setup(row):
    return choosen_explanation_prompt.format(row['question'],row['chosen'])
cloud_dpo['choosen_explanation_prompt'] = cloud_dpo.apply(choosen_explanation_prompt_setup, axis=1)


### Rejected Prompt -> Reject answer 

In [89]:
reject_limitations_prompt = '''
Given a question and a rejected answer, explain the limitations or flaws that make this answer inadequate or incorrect.

Question:
{}

Rejected Answer:
{}

Explanation:
'''

In [90]:
def reject_limitations_prompt_setup(row):
    return reject_limitations_prompt.format(row['question'],row['rejected'])
cloud_dpo['rejected_limitations_prompt'] = cloud_dpo.apply(reject_limitations_prompt_setup, axis=1)


## Device

### Choosen Prompt -> Human Answer

In [27]:
choosen_explanation_prompt = '''
Given a question and its selected answer, provide a brief explanation of why this answer is correct and appropriately addresses the question.

Question:
{}

Chosen Answer:
{}

Explanation:
'''

In [29]:
def choosen_explanation_prompt_setup(row):
    return choosen_explanation_prompt.format(row['question'],row['chosen'])
device_dpo['choosen_explanation_prompt'] = device_dpo.apply(choosen_explanation_prompt_setup, axis=1)


### Rejected Prompt -> Reject answer 

In [30]:
reject_limitations_prompt = '''
Given a question and a rejected answer, explain the limitations or flaws that make this answer inadequate or incorrect.

Question:
{}

Rejected Answer:
{}

Explanation:
'''

In [31]:
def reject_limitations_prompt_setup(row):
    return reject_limitations_prompt.format(row['question'],row['rejected'])
device_dpo['rejected_limitations_prompt'] = device_dpo.apply(reject_limitations_prompt_setup, axis=1)


# Generate Response

## cloud setup

In [91]:
def cloud_inference(prompt, model_deployment_name="gpt-4.1"):
    client = load_azure_openai_client()

    response = client.chat.completions.create(
        model=model_deployment_name,  # this must match the deployment name in Azure
        messages=[
            {"role": "user", "content": prompt}
        ],
        seed=42,
        max_tokens=256,
   
    )
    return response.choices[0].message.content


In [92]:
from tqdm import tqdm

def generate_cloud_dpo_responses(df, prompt_column: str, output_column: str, inference_fn=cloud_inference):
    responses = []

    for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
        prompt = row[prompt_column]

        try:
            response = inference_fn(prompt)
        except Exception as e:
            print(f"[Error] Row {idx}: {e}")
            response = None  
        
        responses.append(response)

        # Show first row as sample if successful or failed
        if idx == 0:
            print("\nSample Prompt:\n", prompt)
            print("\nSample Response:\n", response)

    df[output_column] = responses
    return df


## device setup

In [93]:
def device_inference(prompt, model_deployment_name="gpt-4.1"):
    client = load_azure_openai_client()

    response = client.chat.completions.create(
        model=model_deployment_name,  # this must match the deployment name in Azure
        messages=[
            {"role": "user", "content": prompt}
        ],
        seed=42,
        max_tokens=256,
   
    )
    return response.choices[0].message.content


In [94]:
from tqdm import tqdm

def generate_device_dpo_responses(df, prompt_column: str, output_column: str, inference_fn=device_inference):
    responses = []

    for idx, row in tqdm(df.iterrows(), total=df.shape[0]):
        prompt = row[prompt_column]

        try:
            response = inference_fn(prompt)
        except Exception as e:
            print(f"[Error] Row {idx}: {e}")
            response = None  

        responses.append(response)

        # Show first row as sample
        if idx == 0:
            print("\nSample Prompt:\n", prompt)
            print("\nSample Response:\n", response)

    df[output_column] = responses
    return df


## Testing Samples

In [95]:
sample =cloud_dpo[10:15]

In [96]:
sample.head(2)

,question,chosen,rejected,choosen_explanation_prompt,rejected_limitations_prompt
10,OCR Text:\nimg0:\nNon-table text:\n ...,\r\nThere are two settings here that affect th...,This property has no effect on consumer side.\...,"\nGiven a question and its selected answer, pr...","\nGiven a question and a rejected answer, expl..."
11,OCR Text:\nimg0:\nNon-table text:\n ...,\r\nYou can add a powershell task at the end o...,The problem here isn't really about how to dis...,"\nGiven a question and its selected answer, pr...","\nGiven a question and a rejected answer, expl..."


In [97]:
#print(sample['choosen_explanation_prompt'][0])

In [98]:
sample = generate_cloud_dpo_responses(sample,prompt_column='choosen_explanation_prompt',output_column='generated_choosen_explanation')


  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:22<00:00,  4.55s/it]
C:\Users\s\AppData\Local\Temp\ipykernel_5336\2169679680.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[output_column] = responses


In [99]:

sample = generate_cloud_dpo_responses(sample,prompt_column='rejected_limitations_prompt',output_column='generated_rejected_limitations')


100%|██████████| 5/5 [00:26<00:00,  5.24s/it]
C:\Users\s\AppData\Local\Temp\ipykernel_5336\2169679680.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[output_column] = responses


In [102]:
print(sample['generated_choosen_explanation'][11])

**Explanation:**

This answer is correct because it directly addresses the problem described: warnings in the linting step appear in the output, but do not impact the build status, which remains "success" even if warnings are present. The solution uses a PowerShell script that calls the Azure DevOps REST API to inspect the build timeline for warnings after the pipeline has run. If any warnings are found, it emits the specific Azure DevOps logging command (`##vso[task.complete result=SucceededWithIssues;]`), which sets the build status to "SucceededWithIssues" rather than plain "Succeeded." 

This status change flags the build as having issues (warnings), so you can easily spot builds that require attention, even if they completed. The use of the REST API is necessary because there is no built-in Azure DevOps feature to automatically set the build status based on warning presence in tasks. Thus, the answer provides both an explanation and a working sample script to achieve the desired b

In [45]:
sample.head(2)

,question,chosen,rejected,choosen_explanation_prompt,rejected_limitations_prompt,choosen_explanation_responses,generated_choosen_explanation,generated_rejected_limitations
10,OCR Text:\nimg0:\nNon-table text:\n ...,\r\nThere are two settings here that affect th...,This property has no effect on consumer side.\...,"\nGiven a question and its selected answer, pr...","\nGiven a question and a rejected answer, expl...",None,**Explanation:**\n\nThe chosen answer correctl...,Here’s an explanation of the flaws and limitat...
11,OCR Text:\nimg0:\nNon-table text:\n ...,\r\nYou can add a powershell task at the end o...,The problem here isn't really about how to dis...,"\nGiven a question and its selected answer, pr...","\nGiven a question and a rejected answer, expl...",None,**Explanation:**\n\nThe chosen answer is corre...,**Limitations and flaws of the rejected answer...


In [85]:
print(sample['generated_rejected_limitations'].loc[11])

The rejected answer is limited because it misunderstands both Azure AD B2C’s behavior and the core security concern. 

- **Incorrect Design Justification:** The answer claims redirecting to arbitrary URLs after logout is “by design,” supposedly to protect against CSRF or to restrict to the original client’s domain. This is inaccurate—Azure AD B2C is intended to restrict post-logout redirects to pre-registered, trusted redirect URIs. Allowing arbitrary redirects is not “by design” and directly exposes users to open redirect vulnerabilities.
- **Misses the Security Risk:** Open redirects are a well-known vulnerability type that can be exploited for phishing or token theft; the answer downplays this risk and provides misleading assurance.
- **Irrelevant Cookie Scenario:** The cookie theft example and logic do not apply, since the main concern is attackers using crafted URLs to hijack user flows.

In summary


In [88]:
sample.to_csv("sample-dpo-response.csv")

## Cloud Testset

In [103]:
cloud_dpo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2349 entries, 0 to 2348
Data columns (total 5 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   question                     2349 non-null   object
 1   chosen                       2349 non-null   object
 2   rejected                     2349 non-null   object
 3   choosen_explanation_prompt   2349 non-null   object
 4   rejected_limitations_prompt  2349 non-null   object
dtypes: object(5)
memory usage: 91.9+ KB


In [104]:
cloud_dpo.head(2)

,question,chosen,rejected,choosen_explanation_prompt,rejected_limitations_prompt
0,OCR Text:\nimg0:\nNon-table text:\n ...,A service connection is a connector to an exte...,No answer,"\nGiven a question and its selected answer, pr...","\nGiven a question and a rejected answer, expl..."
1,OCR Text:\nimg0:\nNon-table text:\n ...,\r\nYour error cognito-idp:ListUsers is about ...,The issue here seems that your role does not h...,"\nGiven a question and its selected answer, pr...","\nGiven a question and a rejected answer, expl..."


### Choose Explanation Responses

In [105]:
cloud_dpo_responses = generate_cloud_dpo_responses(cloud_dpo,prompt_column='choosen_explanation_prompt',output_column='choosen_explanation_responses')

  0%|          | 1/2349 [00:04<2:44:17,  4.20s/it]


Sample Prompt:
 
Given a question and its selected answer, provide a brief explanation of why this answer is correct and appropriately addresses the question.

Question:
OCR Text:
img0:
Non-table text:
                                                                                                                                                                                                                                                                                             о
                                                                                                                                                                                                                                                                                                      ☑
                                                                                                                                                                                                                      

 84%|████████▍ | 1973/2349 [2:08:51<20:18,  3.24s/it]    

[Error] Row 1972: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': True, 'detected': True}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}


100%|██████████| 2349/2349 [2:29:39<00:00,  3.82s/it]


In [106]:
cloud_dpo_responses['choosen_explanation_responses'].loc[98]

"**Explanation:**\n\nThe chosen answer correctly outlines the configuration steps needed to use Google Apps SMTP relay (`smtp-relay.gmail.com`) with Postfix on a Google Compute Engine instance. It addresses the question's requirements and the issues described in the error message:\n\n1. **Uses the correct SMTP relay and port:** It sets `relayhost = [smtp-relay.gmail.com]:587` (since port 25 is blocked on Google Compute Engine).\n2. **Configures proper domain presentation:** By setting `smtp_helo_name` to your Google Apps domain, it ensures the HELO/EHLO command matches your registered domain, as required by Google's SMTP relay policy (referenced in the error message).\n3. **Enables TLS encryption:** The configuration enables TLS, which is required for secure communication with Google's SMTP relay.\n4. **Limits interface and protocol:** The loopback and IPv4 settings match Google Compute's networking restrictions.\n5. **Handles mail origin:** Ensures messages are sent with the correct e

In [43]:
#cloud_dpo=pd.read_csv("cloud_dpo_generated_response_no_answer_100_with_ocr.csv")

### Reject Limitations Responses

In [107]:
cloud_dpo_responses = generate_cloud_dpo_responses(cloud_dpo,prompt_column='rejected_limitations_prompt',output_column='rejected_limitations_responses')

  0%|          | 1/2349 [00:04<2:49:44,  4.34s/it]


Sample Prompt:
 
Given a question and a rejected answer, explain the limitations or flaws that make this answer inadequate or incorrect.

Question:
OCR Text:
img0:
Non-table text:
                                                                                                                                                                                                                                                                                             о
                                                                                                                                                                                                                                                                                                      ☑
                                                                                                                                                                                                                                            

 84%|████████▍ | 1973/2349 [2:15:58<26:24,  4.21s/it]  

[Error] Row 1972: Error code: 400 - {'error': {'message': "The response was filtered due to the prompt triggering Azure OpenAI's content management policy. Please modify your prompt and retry. To learn more about our content filtering policies please read our documentation: https://go.microsoft.com/fwlink/?linkid=2198766", 'type': None, 'param': 'prompt', 'code': 'content_filter', 'status': 400, 'innererror': {'code': 'ResponsibleAIPolicyViolation', 'content_filter_result': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': True, 'detected': True}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}}}}}


100%|██████████| 2349/2349 [2:42:38<00:00,  4.15s/it]


In [112]:
cloud_dpo_responses.dropna(inplace=True)

In [113]:
cloud_dpo_responses.to_csv("cloud_dpo_train_generated_responses.csv", index=False)

In [114]:
cloud_dpo_responses.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2348 entries, 0 to 2348
Data columns (total 7 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   question                        2348 non-null   object
 1   chosen                          2348 non-null   object
 2   rejected                        2348 non-null   object
 3   choosen_explanation_prompt      2348 non-null   object
 4   rejected_limitations_prompt     2348 non-null   object
 5   choosen_explanation_responses   2348 non-null   object
 6   rejected_limitations_responses  2348 non-null   object
dtypes: object(7)
memory usage: 146.8+ KB


In [115]:
cloud_dpo_responses.head(2)

,question,chosen,rejected,choosen_explanation_prompt,rejected_limitations_prompt,choosen_explanation_responses,rejected_limitations_responses
0,OCR Text:\nimg0:\nNon-table text:\n ...,A service connection is a connector to an exte...,No answer,"\nGiven a question and its selected answer, pr...","\nGiven a question and a rejected answer, expl...",The chosen answer is correct because it explai...,**Limitations/Flaws with the Rejected Answer (...
1,OCR Text:\nimg0:\nNon-table text:\n ...,\r\nYour error cognito-idp:ListUsers is about ...,The issue here seems that your role does not h...,"\nGiven a question and its selected answer, pr...","\nGiven a question and a rejected answer, expl...",The chosen answer is correct because it identi...,**Limitations and flaws of the rejected answer...


In [ ]:
#cloud_dpo_responses.to_csv("cloud_dpo_generated_no_answers_responses_100_with_ocr.csv", index=False)

In [47]:
print(cloud_dpo_responses['choosen_explanation_responses'].loc[1])

Explanation:

The chosen answer is "No answer" because the question was marked as **closed** due to being unclear, and not open for answers. On many Q&A platforms, when a question is closed for being too broad or unclear, users are prevented from submitting responses. Therefore, "No answer" appropriately addresses the question as no valid or accepted answer can be provided while the post remains closed.


In [49]:
print(cloud_dpo_responses['rejected_limitations_responses'].loc[1])

Certainly! I’ll review the **rejected answer** for the initial question about app service configuration and explain its limitations or flaws that make it inadequate or incorrect.

---

**Rejected Answer Summary:**  
The answer interprets the question as about hosting frontend and admin portal separately in Azure App Service, suggests both can be separated, speaks of configurations, and mentions possible use of virtual directories and best practices, then advises consulting documentation or experts due to limited context.

---

## Limitations / Flaws in the Rejected Answer

### 1. **Overgeneralization & Assumptions**
- The answer **assumes** the user is referring specifically to Azure App Service, even though the OCR text and question do not mention Azure. The technologies referenced are only vaguely ".NET or another framework".
- It infers that `/site/sub/admin` and other paths must be separate applications/web apps, but the actual question is ambiguous and leans more toward how **virt

## Device Testset

In [50]:
device_dpo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 5 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   question                     100 non-null    object
 1   chosen                       100 non-null    object
 2   rejected                     100 non-null    object
 3   choosen_explanation_prompt   100 non-null    object
 4   rejected_limitations_prompt  100 non-null    object
dtypes: object(5)
memory usage: 4.0+ KB


In [52]:
device_dpo.head(2)

,question,chosen,rejected,choosen_explanation_prompt,rejected_limitations_prompt
0,OCR Text: \nimg0: \nNon-table text: \n ...,No answer,The configuration of Azure Functions involves ...,"\nGiven a question and its selected answer, pr...","\nGiven a question and a rejected answer, expl..."
1,OCR Text: \nimg0: \nTables: \nTable 0: \n+...,No answer,"Based on the provided OCR-text, I cannot deter...","\nGiven a question and its selected answer, pr...","\nGiven a question and a rejected answer, expl..."


### Choose Explanation Responses

In [53]:
device_dpo_responses = generate_device_dpo_responses(device_dpo,prompt_column='choosen_explanation_prompt',output_column='choosen_explanation_responses')


  1%|          | 1/100 [00:03<05:12,  3.15s/it]


Sample Prompt:
 
Given a question and its selected answer, provide a brief explanation of why this answer is correct and appropriately addresses the question.

Question:
OCR Text:  
img0:  
Non-table text:  
                                                                                    | ?  
                                       Doc  
                                S  
                                 ~  
                           D  
=  
    Cloud Something?  
                                                                            ✓  
                Search things and docs (G+/)  
        > BlobThing1  
Home >  
     BlobThing1 | Code+Something  
                                                                           ☑  
</>  
  Function  
 Search (Ctrl+/)  
                                 Save X Discard  
                                         Refresh  
                                               Test?/Run Upload  
                    «  
(fx} Summary  
        

100%|██████████| 100/100 [04:37<00:00,  2.77s/it]


In [54]:
device_dpo_responses.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   question                       100 non-null    object
 1   chosen                         100 non-null    object
 2   rejected                       100 non-null    object
 3   choosen_explanation_prompt     100 non-null    object
 4   rejected_limitations_prompt    100 non-null    object
 5   choosen_explanation_responses  100 non-null    object
dtypes: object(6)
memory usage: 4.8+ KB


### Reject Limitations Responses

In [55]:
device_dpo_responses = generate_device_dpo_responses(device_dpo,prompt_column='rejected_limitations_prompt',output_column='rejected_limitations_responses')

  1%|          | 1/100 [00:04<07:46,  4.71s/it]


Sample Prompt:
 
Given a question and a rejected answer, explain the limitations or flaws that make this answer inadequate or incorrect.

Question:
OCR Text:  
img0:  
Non-table text:  
                                                                                    | ?  
                                       Doc  
                                S  
                                 ~  
                           D  
=  
    Cloud Something?  
                                                                            ✓  
                Search things and docs (G+/)  
        > BlobThing1  
Home >  
     BlobThing1 | Code+Something  
                                                                           ☑  
</>  
  Function  
 Search (Ctrl+/)  
                                 Save X Discard  
                                         Refresh  
                                               Test?/Run Upload  
                    «  
(fx} Summary  
                              

100%|██████████| 100/100 [07:51<00:00,  4.72s/it]


In [57]:
device_dpo_responses.head(2)

,question,chosen,rejected,choosen_explanation_prompt,rejected_limitations_prompt,choosen_explanation_responses,rejected_limitations_responses
0,OCR Text: \nimg0: \nNon-table text: \n ...,No answer,The configuration of Azure Functions involves ...,"\nGiven a question and its selected answer, pr...","\nGiven a question and a rejected answer, expl...","Explanation:\n\nThe selected answer is ""No ans...",**Explanation of Limitations/Flaws in the Reje...
1,OCR Text: \nimg0: \nTables: \nTable 0: \n+...,No answer,"Based on the provided OCR-text, I cannot deter...","\nGiven a question and its selected answer, pr...","\nGiven a question and a rejected answer, expl...","Explanation:\n\nThe selected answer is ""No ans...",**Limitations and Flaws in the Rejected Answer...


In [59]:
device_dpo_responses.to_csv("device_dpo_generated_no_answers_responses_100_with_ocr.csv",index=False)